# msgs

> Canonical messages and wire conversions.

A conversation is a list of OpenAI-style dicts. This shape survives JSON round-trips and requires no custom type to inspect.

Callers can pass text, media paths, or lists of both. `mk_msg` normalizes that input. `to_oai_msg` converts a message for the wire. `norm_resp` converts a reply back.

In [ ]:
#| default_exp msgs

In [ ]:
#| export
import json, math, os
from base64 import b64encode
from fastcore.funccall import get_schema
from fastcore.all import Path, L, first, listify, detect_mime
from aidialog.msg_parts import InputImage, InputAudio, data_url
from urai.core import Resp, resp_text
from urai.tags import split_think, parse_tool_tags

In [ ]:
#| hide
from base64 import b64decode
from fastcore.test import test_eq, test_fail
tiny_png = b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8'
                     'z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg==')
tiny_wav = (b'RIFF$\x00\x00\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x44\xac\x00\x00'
            b'\x88X\x01\x00\x02\x00\x10\x00data\x00\x00\x00\x00')

## Content parts

One content part. A `str` is text, `bytes` or a `Path` is sniffed for its type, and a dict is already a part and passes through untouched. Anything that is neither text, an image, nor audio raises rather than being dropped, because a caller who attached a PDF wants to hear about it.

In [ ]:
#| export
_audio_fmts = {'audio/wav': 'wav', 'audio/x-wav': 'wav', 'audio/wave': 'wav',
               'audio/mpeg': 'mp3', 'audio/mp3': 'mp3'}

def _audio_fmt(mime):
    "OpenAI `input_audio` format name for a MIME type."
    return _audio_fmts.get(mime, mime.split('/')[-1])

def mk_content(o):
    "`o` as an OpenAI-style content part: text, a base64 `image_url`, or an `input_audio`."
    if isinstance(o, dict): return o
    if isinstance(o, str): return {'type': 'text', 'text': o}
    if isinstance(o, os.PathLike): o = Path(o).read_bytes()
    if isinstance(o, bytes):
        mime = detect_mime(o) or 'application/octet-stream'
        if mime.startswith('image/'):
            return {'type': 'image_url',
                    'image_url': {'url': f'data:{mime};base64,{b64encode(o).decode()}'}}
        if mime.startswith('audio/'):
            return {'type': 'input_audio',
                    'input_audio': {'data': b64encode(o).decode(), 'format': _audio_fmt(mime)}}
        raise TypeError(f'chat supports text, image, and audio content, got {mime}')
    raise TypeError(f'Unsupported content type: {type(o)}')

def is_media(p):
    "Is `p` an image or audio content part?"
    return isinstance(p, dict) and p.get('type') in ('image_url', 'input_audio')

In [ ]:
test_eq(mk_content('hi'), {'type': 'text', 'text': 'hi'})
test_eq(mk_content({'type': 'text', 'text': 'passthrough'}), {'type': 'text', 'text': 'passthrough'})
test_eq(mk_content(tiny_png)['type'], 'image_url')
assert mk_content(tiny_png)['image_url']['url'].startswith('data:image/png;base64,')
test_eq(mk_content(tiny_wav)['input_audio']['format'], 'wav')

In [ ]:
test_fail(lambda: mk_content(b'%PDF-1.4 not media'), contains='text, image, and audio')
test_fail(lambda: mk_content(42), contains='Unsupported content type')
test_eq(is_media(mk_content(tiny_png)), True)
test_eq(is_media(mk_content('hi')), False)
test_eq(_audio_fmt('audio/mpeg'), 'mp3')
test_eq(_audio_fmt('audio/ogg'), 'ogg')     # unknown: the subtype is the best guess

## Messages

A message of all-text parts collapses back to a plain string, because that is what every transport prefers and what a reader wants to see in a history. Only a message that actually carries media keeps the list form.

In [ ]:
#| export
def mk_msg(content, role='user'):
    "An OpenAI-style message dict from a str, bytes, list, or dict."
    if content is None or isinstance(content, dict): return content
    parts = [mk_content(o) for o in content] if isinstance(content, list) else [mk_content(content)]
    if all(p.get('type') == 'text' for p in parts):
        return {'role': role, 'content': '\n'.join(p['text'] for p in parts)}
    return {'role': role, 'content': parts}

def mk_msgs(msgs):
    "Normalize a list of messages to OpenAI-style dicts."
    return [mk_msg(m) for m in listify(msgs)] if msgs else []

In [ ]:
test_eq(mk_msg('hi'), {'role': 'user', 'content': 'hi'})
test_eq(mk_msg(['a', 'b']), {'role': 'user', 'content': 'a\nb'})     # all text, so collapsed
test_eq(mk_msg('ok', 'assistant'), {'role': 'assistant', 'content': 'ok'})
test_eq(mk_msg(None), None)

In [ ]:
m = mk_msg(['look:', tiny_png])
test_eq(m['role'], 'user')
test_eq([p['type'] for p in m['content']], ['text', 'image_url'])     # media keeps the list
test_eq(mk_msgs(['a', 'b']), [{'role': 'user', 'content': 'a'}, {'role': 'user', 'content': 'b'}])
test_eq(mk_msgs(None), [])

Past-turn media is re-sent on every subsequent call otherwise, which costs tokens on every turn for a picture the model already described. `strip_media` swaps it for a placeholder, so the history says a picture was there without carrying it again.

In [ ]:
#| export
_media_ph = {'image_url': '[image]', 'input_audio': '[audio]'}

def strip_media(m):
    "Replace media parts with a text placeholder and collapse content to a string."
    c = m.get('content')
    if not isinstance(c, list): return m
    return {**m, 'content': '\n'.join(_media_ph.get(p.get('type'), '[media]') if is_media(p)
                                      else p.get('text', '') for p in c)}

In [ ]:
test_eq(strip_media(mk_msg(['look:', tiny_png])), {'role': 'user', 'content': 'look:\n[image]'})
test_eq(strip_media(mk_msg(['hear:', tiny_wav])), {'role': 'user', 'content': 'hear:\n[audio]'})
test_eq(strip_media({'role': 'user', 'content': 'plain'}), {'role': 'user', 'content': 'plain'})

Backends that speak aidialog's typed parts rather than dicts convert at their own edge.

In [ ]:
#| export
def to_media_part(p):
    "An OpenAI-style media content part as a typed aidialog `Part`, or None if `p` is not media."
    if not is_media(p): return None
    if p['type'] == 'image_url':
        image = p.get('image_url')
        url = image.get('url') if isinstance(image, dict) else image
        return InputImage(text=url, mime=(data_url(url) or (None, None))[0])
    audio = p.get('input_audio') or {}
    url = f"data:audio/{audio.get('format', 'wav')};base64,{audio.get('data', '')}"
    return InputAudio(text=url, mime=(data_url(url) or (None, None))[0])

In [ ]:
test_eq(to_media_part(mk_content(tiny_png)).mime, 'image/png')
test_eq(to_media_part(mk_content(tiny_wav)).mime, 'audio/wav')
test_eq(to_media_part({'type': 'text', 'text': 'hi'}), None)

## Tools

A Python function becomes a tool schema through `fastcore.funccall.get_schema`. A dict passes through as an existing schema. Dict schemas also describe tools that a provider runs itself.

In [ ]:
#| export
def mk_toolspec(f):
    "OpenAI-style tool spec for callable `f`. Spec dicts pass through."
    if isinstance(f, dict): return f
    sc = get_schema(f, pname='parameters')
    sc.get('parameters', {}).pop('title', None)   # get_schema adds a stray title for classes
    return {'type': 'function', 'function': sc}

In [ ]:
def add(a: int,      # first addend
        b: int = 2   # second addend
       ) -> int:
    "Add two numbers."
    return a + b

spec = mk_toolspec(add)
test_eq(spec['type'], 'function')
test_eq(spec['function']['name'], 'add')
assert spec['function']['description'].startswith('Add two numbers.')
test_eq(sorted(spec['function']['parameters']['properties']), ['a', 'b'])
test_eq(mk_toolspec(spec), spec)      # already a spec

## Tool calls

`ToolCall` remains a `dict` for existing indexing code and adds named accessors. `server=True` marks a call already run by the provider. The loop records server calls but never executes them.

In [ ]:
#| export
import uuid

class ToolCall(dict):
    "One tool call in canonical form. A `dict` subclass, so anything that indexes it keeps working."
    def __init__(self, name='', arguments=None, id=None, server=False):
        super().__init__(id=id or f'call_{uuid.uuid4().hex[:8]}', type='function',
                         function={'name': name, 'arguments': arguments if arguments is not None else {}})
        if server: self['server'] = True
    @property
    def name(self): return self['function'].get('name', '')
    @property
    def arguments(self): return self['function'].get('arguments') or {}
    @property
    def server(self):
        "True when the provider runs this tool itself, so the loop must not."
        return bool(self.get('server'))
    def __repr__(self): return f'ToolCall({self.name}, {self.arguments})'

def tc_name(tc):
    "Name of a tool call, however it was built."
    return (tc.get('function') or {}).get('name', '')

def mk_tool_res_msg(tc, result):
    "Canonical `role='tool'` message carrying `result` back for tool call `tc`."
    return {'role': 'tool', 'tool_call_id': tc.get('id'), 'name': tc_name(tc), 'content': str(result)}

def mk_tool_res_msgs(tcs, results):
    "Canonical tool-result messages for several calls at once."
    return [mk_tool_res_msg(tc, r) for tc, r in zip(tcs, results)]

In [ ]:
tc = ToolCall('add', {'a': 1})
test_eq((tc.name, tc.arguments, tc.server), ('add', {'a': 1}, False))
test_eq(repr(tc), 'ToolCall(add, {\'a\': 1})')
test_eq(tc['type'], 'function')
test_eq(tc_name(tc), 'add')
test_eq(ToolCall('web_search', id='s1', server=True).server, True)
test_eq(ToolCall().arguments, {})

In [ ]:
test_eq(mk_tool_res_msg(ToolCall('add', id='c1'), 3),
        {'role': 'tool', 'tool_call_id': 'c1', 'name': 'add', 'content': '3'})
test_eq(len(mk_tool_res_msgs([ToolCall('a'), ToolCall('b')], [1, 2])), 2)
test_eq(mk_tool_res_msgs([ToolCall('a')], []), [])     # zip stops at the shorter side

## In from the wire

`norm_resp` converts a provider completion to a Urai reply. Tagged and native calls both become `tool_calls`.

Tool arguments arrive as JSON strings. Malformed JSON becomes an empty dict. The tool loop can report a bad call to the model, but it cannot recover from a parser exception mid-turn.

In [ ]:
#| export
def parse_args(a):
    "Parse OpenAI JSON-string tool arguments to a dict. Dicts pass through, bad JSON becomes `{}`."
    if isinstance(a, dict): return a
    try: return json.loads(a) if a else {}
    except json.JSONDecodeError: return {}

def norm_resp(r):
    "An OpenAI-style chat completion as a urai `Resp`."
    ch = r['choices'][0]
    m = ch.get('message') or {}
    text, th = split_think(m.get('content') or '')
    text, tag_tcs = parse_tool_tags(text)
    tcs = [{'id': tc.get('id'), 'type': 'function',
            'function': {'name': tc.get('function', {}).get('name', ''),
                         'arguments': parse_args(tc.get('function', {}).get('arguments'))}}
           for tc in (m.get('tool_calls') or [])] + tag_tcs
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if tcs: res['tool_calls'] = tcs
    if ch.get('finish_reason') == 'length': res['truncated'] = True
    if 'usage' in r: res['usage'] = dict(r['usage'])
    return Resp(res)

In [ ]:
test_eq(parse_args('{"a": 1}'), {'a': 1})
test_eq(parse_args({'a': 1}), {'a': 1})
test_eq(parse_args(''), {})
test_eq(parse_args('{oops'), {})      # a bad call is reportable; a crash mid-turn is not

In [ ]:
r = norm_resp({'choices': [{'message': {'content': '<think>hmm</think>the answer'}}],
               'usage': {'total_tokens': 9}})
test_eq(r['content'], 'the answer')
test_eq(r['channels'], {'thought': 'hmm'})
test_eq(r['usage'], {'total_tokens': 9})
test_eq('tool_calls' in r, False)
test_eq(type(r), Resp)

In [ ]:
r = norm_resp({'choices': [{'finish_reason': 'length', 'message': {
    'content': '', 'tool_calls': [{'id': 'c1', 'function': {'name': 'add', 'arguments': '{"a": 1}'}}]}}]})
test_eq(r['tool_calls'], [{'id': 'c1', 'type': 'function',
                           'function': {'name': 'add', 'arguments': {'a': 1}}}])
test_eq(r['truncated'], True)

In [ ]:
# a native call and a tagged one in the same reply both land in `tool_calls`
r = norm_resp({'choices': [{'message': {
    'content': '<tool_call>{"name": "b", "arguments": {}}</tool_call>',
    'tool_calls': [{'id': 'c1', 'function': {'name': 'a', 'arguments': '{}'}}]}}]})
test_eq([t['function']['name'] for t in r['tool_calls']], ['a', 'b'])

## Out to the wire

The history carries keys a provider would reject, so `to_oai_msg` projects out just the five it wants. Tool-call arguments go back to JSON strings, and a call turn gets an explicit empty `content` because several providers reject the message without one.

In [ ]:
#| export
def to_oai_msg(m):
    "Convert a history entry to an OpenAI wire message."
    out = {k: m[k] for k in ('role', 'content', 'tool_calls', 'tool_call_id', 'name')
           if m.get(k) is not None}
    if 'tool_calls' in out:
        out['tool_calls'] = [
            {'id': tc.get('id') or f'call_{i}', 'type': 'function',
             'function': {'name': tc['function'].get('name', ''),
                          'arguments': a if isinstance((a := tc['function'].get('arguments')), str)
                                       else json.dumps(a or {})}}
            for i, tc in enumerate(out['tool_calls'])]
        out.setdefault('content', '')
    return out

In [ ]:
test_eq(to_oai_msg({'role': 'assistant', 'content': 'hi', 'channels': {'thought': 'x'},
                    'usage': {'total_tokens': 1}}),
        {'role': 'assistant', 'content': 'hi'})       # urai-only keys dropped

In [ ]:
out = to_oai_msg({'role': 'assistant', 'tool_calls': [ToolCall('add', {'a': 1}, id='c1')]})
test_eq(out['content'], '')                            # providers reject a call turn with no content
test_eq(out['tool_calls'][0]['function']['arguments'], '{"a": 1}')
test_eq(to_oai_msg({'role': 'assistant', 'tool_calls': [{'function': {'name': 'f'}}]}
                   )['tool_calls'][0]['id'], 'call_0')  # a missing id is filled in by position

## Usage across rounds

One turn can contain several wire calls. `sum_usage` combines their usage. An unreported hosted metric stays absent rather than becoming a measured zero.

In [ ]:
#| export
def sum_usage(us):
    "Sum OpenAI usage dicts, ignoring Nones. `model` is carried through rather than summed."
    us = [u for u in us if u]
    if not us: return None
    out = {k: sum(u.get(k, 0) for u in us)
           for k in ('prompt_tokens', 'completion_tokens', 'total_tokens', 'cached_tokens')}
    for k in ('reasoning_tokens', 'cache_creation_tokens', 'cost'):   # hosted-only: absent, not 0
        if (v := sum(u.get(k, 0) for u in us)): out[k] = v
    if (m := first(us, lambda u: u.get('model'))): out['model'] = m['model']
    return out

In [ ]:
test_eq(sum_usage([{'total_tokens': 3}, None, {'total_tokens': 4, 'model': 'gpt-5.1'}]),
        {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 7,
         'cached_tokens': 0, 'model': 'gpt-5.1'})
test_eq(sum_usage([]), None)
test_eq(sum_usage([None]), None)
test_eq('cost' in sum_usage([{'cost': 0.0}]), False)   # nothing charged, so nothing claimed
test_eq(sum_usage([{'cost': 0.5}])['cost'], 0.5)

## Prompt transports

Some backends take one block of text rather than a message list, and have no tokenizer to lend. These two cover both gaps: a rough token count calibrated on small local models, and a rendering of the history that keeps tool calls visible in the same tag form the model was taught.

In [ ]:
#| export
CHARS_PER_TOKEN = 3.25   #: measured on ornith-1.5:9b and qwen3:0.6b, which both give 3.50

def est_tokens(text):
    "Rough token count for a transport with no tokenizer of its own."
    return max(1, math.ceil(len(text) / CHARS_PER_TOKEN)) if text else 0

#: How each role is titled when a conversation is rendered as one block of text.
ROLE_NAMES = {'user': 'User', 'assistant': 'Assistant', 'tool': 'Tool result', 'system': 'System'}

def render_prompt(hist, sp=''):
    "A conversation as one block of text, for a transport that takes a prompt rather than messages."
    out = [sp] if sp else []
    for m in hist:
        if not (txt := resp_text(m)) and not m.get('tool_calls'): continue
        who = ROLE_NAMES.get(m.get('role'), m.get('role', '?'))
        if m.get('role') == 'tool': who = f"Tool result ({m.get('name', '?')})"
        calls = '\n'.join(
            '<tool_call>\n' + json.dumps({'name': tc_name(tc),
                                          'arguments': tc.get('function', {}).get('arguments') or {}})
            + '\n</tool_call>' for tc in (m.get('tool_calls') or []))
        out.append(f"## {who}\n{txt}{chr(10) + calls if calls else ''}")
    return '\n\n'.join(out)

In [ ]:
test_eq(est_tokens(''), 0)
test_eq(est_tokens('a'), 1)
test_eq(est_tokens('a' * 13), 4)

In [ ]:
hist = [{'role': 'user', 'content': 'what is 1+1?'},
        {'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('add', {'a': 1, 'b': 1})]},
        {'role': 'tool', 'name': 'add', 'content': '2'},
        {'role': 'assistant', 'content': 'Two.'}]
print(render_prompt(hist, 'Be brief.'))

In [ ]:
out = render_prompt(hist, 'Be brief.')
test_eq(out.split('\n\n')[0], 'Be brief.')
assert '## Tool result (add)' in out and '<tool_call>' in out
test_eq(render_prompt([{'role': 'user', 'content': ''}]), '')   # nothing to say, nothing rendered

## Prefix reuse

A local engine can keep the KV cache for however much of the new prompt matches the old one. Backends compare token lists with this to find out how much.

In [ ]:
#| export
def common_prefix_len(a, b):
    "Length of the longest common prefix of sequences `a` and `b`."
    n = 0
    for x, y in zip(a, b):
        if x != y: break
        n += 1
    return n

In [ ]:
test_eq(common_prefix_len([1, 2, 3, 4], [1, 2, 9]), 2)
test_eq(common_prefix_len([1, 2], [1, 2]), 2)
test_eq(common_prefix_len([], [1]), 0)
test_eq(common_prefix_len([9], [1]), 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()